# ML-03 — Frame My Lane as an ML Task

**Lane 2: Refresh / Content Opportunity Scoring.** This notebook maps the lane onto a decision, target, metric, action, and real one-row-per-page dataframe before any model is trained.

## 1. My lane as an ML task (type)

**Primary task type: ranking / scoring.** The real question is “which pages should a content strategist review first?” I will assign each eligible page an explainable opportunity/risk score and sort the pages into a top-20 review queue. A binary classifier may later estimate one component of that score—the probability of a future visibility decline—but the operational output is a ranking, not a yes/no command.

- **Decision improved:** how to allocate the next 20 page-review slots.
- **Who acts and how:** a content strategist examines the ranked evidence and chooses to protect, refresh, expand, consolidate, prune, or monitor a page.
- **Cost of a false positive:** review/editor time is wasted, and an unnecessary edit could damage a healthy page.
- **Cost of a false negative:** a valuable page losing visibility may wait too long for review.

Because a false positive can trigger costly work, the model supports a human decision; it never automatically edits or removes content.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

candidate_paths = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
]
data_path = next((path for path in candidate_paths if path.exists()), None)
if data_path is None:
    raise FileNotFoundError("Starter CSV not found. Run from the repo root or work/notebooks.")

starter = pd.read_csv(data_path)
required_fields = {
    "content_id",
    "client_id",
    "impressions_90d",
    "clicks_90d",
    "avg_position",
    "ctr",
    "content_age_days",
    "days_since_last_update",
    "engagement_rate",
    "trend_direction",
    "trend_pct",
}
missing_fields = required_fields.difference(starter.columns)
assert not missing_fields, f"Missing required fields: {sorted(missing_fields)}"
assert starter["content_id"].is_unique

print(f"Starter data loaded: {len(starter):,} rows x {starter.shape[1]} columns")
print(f"Unit check: {starter['content_id'].nunique():,} unique pseudonymized pages across {starter['client_id'].nunique()} clients")

Starter data loaded: 30,000 rows x 44 columns
Unit check: 30,000 unique pseudonymized pages across 32 clients


## 2. Target or proxy

**Desired later target:** an observed change in search visibility measured in a future warehouse window that does not overlap the feature window. The later data contract will define the exact dates, coverage guard, and minimum-volume rule before modeling.

**Starter-data teaching proxy:** `is_declining_proxy = 1` when `trend_direction == "down"`. This is useful for sketching the shape of a binary relevance column, but it is **not an independent future outcome**: `trend_direction` is a rule derived from `trend_pct` in the same snapshot. Therefore this notebook makes no predictive-performance claim from it, and both `trend_direction` and `trend_pct` are forbidden as features.

The eventual ranking score will represent directional evidence that a page deserves review—not a promise that editing it will cause recovery.

In [2]:
# Sketch the proxy column without pretending it is an honest future label.
starter["is_declining_proxy"] = starter["trend_direction"].str.lower().eq("down").astype("int8")

target_contract = pd.DataFrame(
    {
        "role": ["Desired target", "Starter teaching proxy", "Leakage ban"],
        "definition": [
            "Observed next-window visibility change from a separate future period",
            "1 when current snapshot trend_direction equals down; otherwise 0",
            "trend_direction and trend_pct never enter the feature matrix",
        ],
        "use now": ["Framing only", "Column-shape and metric practice only", "Always enforced"],
    }
)
display(target_contract)

proxy_counts = starter["is_declining_proxy"].value_counts().sort_index().rename_axis("is_declining_proxy").to_frame("pages")
proxy_counts["share"] = (proxy_counts["pages"] / len(starter)).round(3)
display(proxy_counts)

,role,definition,use now
0,Desired target,Observed next-window visibility change from a ...,Framing only
1,Starter teaching proxy,1 when current snapshot trend_direction equals...,Column-shape and metric practice only
2,Leakage ban,trend_direction and trend_pct never enter the ...,Always enforced


,pages,share
is_declining_proxy,,
0,13738,0.458
1,16262,0.542


## 3. Success metric

**Primary metric: precision@20.** Of the 20 pages ranked highest for review, what share shows the predeclared future outcome? This matches the team’s fixed review capacity and makes false positives visible.

**What “good” means before training:** on held-out clients and a later time window, at least 15 of the top 20 should meet the outcome (**precision@20 ≥ 0.75**) and the method should beat the predeclared fixed-rule baseline by at least 0.10 absolute. I will also report recall and the outcome base rate so a precise queue cannot hide how many candidates it misses. If ML does not clear those gates consistently, the fixed rule remains the better system.

In [3]:
def precision_at_k(scores, labels, k=20):
    """Share of positive labels among the k highest scores."""
    scores = np.asarray(scores)
    labels = np.asarray(labels)
    if len(scores) != len(labels) or len(scores) < k:
        raise ValueError("scores and labels must have equal length and at least k rows")
    order = np.argsort(-scores, kind="stable")
    return float(labels[order[:k]].mean())

metric_contract = pd.DataFrame(
    {
        "metric": ["Precision@20", "Improvement over fixed rule", "Recall (secondary)"],
        "predeclared bar": [">= 0.75", ">= +0.10 absolute", "Report; no hidden minimum"],
        "why": [
            "At least 15 of 20 scarce review slots are relevant",
            "ML must earn its added complexity",
            "Shows how many outcome pages the queue misses",
        ],
    }
)
display(metric_contract)
assert precision_at_k(np.arange(20), np.ones(20), k=20) == 1.0

,metric,predeclared bar,why
0,Precision@20,>= 0.75,At least 15 of 20 scarce review slots are rele...
1,Improvement over fixed rule,>= +0.10 absolute,ML must earn its added complexity
2,Recall (secondary),Report; no hidden minimum,Shows how many outcome pages the queue misses


## 4. The unit of analysis, as a real dataframe

**One row = one pseudonymized content item (page) in the starter’s trailing-90-day snapshot.** For this lane’s initial decision slice, I keep pages with at least 500 impressions so the review queue focuses on visible pages and avoids treating tiny fluctuations as strong evidence. `content_id` is context only, and `client_id` will later define grouped validation splits; neither is a model feature.

Rate columns such as `ctr` are percentage points (`0.76` means 0.76%), and `avg_position == 0` means missing rather than rank zero, so the display uses `NaN` for that sentinel.

In [4]:
lane_slice = starter.loc[starter["impressions_90d"].ge(500)].copy()
lane_slice["avg_position_for_analysis"] = lane_slice["avg_position"].mask(lane_slice["avg_position"].eq(0))

unit_columns = [
    "content_id",
    "content_type",
    "impressions_90d",
    "clicks_90d",
    "avg_position_for_analysis",
    "ctr",
    "content_age_days",
    "days_since_last_update",
    "is_declining_proxy",
]

unit_summary = pd.DataFrame(
    {
        "check": ["Rows in lane slice", "Unique pages", "Minimum impressions", "Proxy-positive pages"],
        "value": [
            len(lane_slice),
            lane_slice["content_id"].nunique(),
            lane_slice["impressions_90d"].min(),
            int(lane_slice["is_declining_proxy"].sum()),
        ],
    }
)
display(unit_summary)
display(lane_slice[unit_columns].head(8))

assert len(lane_slice) == 16_726
assert lane_slice["content_id"].is_unique
assert lane_slice["impressions_90d"].ge(500).all()
print("PASS: one dataframe row is one unique, visible pseudonymized page.")

,check,value
0,Rows in lane slice,16726
1,Unique pages,16726
2,Minimum impressions,500
3,Proxy-positive pages,9961


,content_id,content_type,impressions_90d,clicks_90d,avg_position_for_analysis,ctr,content_age_days,days_since_last_update,is_declining_proxy
0,content_304f48230142,keyword article,3803,29,10.6,0.76,187,20,1
1,content_a1fb4e703a9e,keyword article,15320,7,20.3,0.05,445,25,1
2,content_9aa793d4d895,keyword article,12581,11,36.5,0.09,141,20,1
3,content_331d6c4de07b,keyword article,11751,58,6.2,0.49,463,22,0
4,content_d99b7a2d90ca,keyword article,19140,24,44.0,0.13,263,14,1
5,content_d4084a4bc775,keyword article,3970,1,8.5,0.03,147,20,1
7,content_a63219c6e95a,keyword article,1724,1,21.2,0.06,445,22,0
8,content_5e6c160719bc,keyword article,32574,29,46.0,0.09,90,20,1


PASS: one dataframe row is one unique, visible pseudonymized page.


## 5. Why ML beats a fixed rule here

A fixed rule such as “old page × at least 500 impressions” is a useful baseline, but it cannot rank most eligible pages: in this slice, the 180-day staleness test flags fewer pages than the 20-slot review capacity and leaves almost the entire inventory tied. The stronger pattern may depend on nonlinear combinations of demand, current position, CTR, age, freshness, content type, and engagement, and those relationships can differ across clients.

ML earns a place only if it ranks held-out clients and later outcomes better than that transparent rule while preserving reason codes. If it does not, I will keep the rule or a dashboard instead.

**One-paragraph frame:** For a content strategist deciding which 20 pages to review next, I will build an explainable ranking from pseudonymized pre-decision content, search, and engagement measurements, scoring later observed visibility risk and evaluating it with precision@20. A wrong call wastes review time or delays attention to a valuable page. A plain rule is not enough because several interacting signals must order thousands of otherwise tied pages. I will claim only observed, directional decision-support—not that a refresh causes recovery.

In [5]:
# Diagnose the transparent baseline before any model is trained.
stale_visible = lane_slice["days_since_last_update"].ge(180)
fixed_rule_score = stale_visible.astype(int) * lane_slice["impressions_90d"]

rule_diagnostic = pd.DataFrame(
    {
        "quantity": [
            "Eligible visible pages",
            "Pages flagged by stale-visible rule",
            "Pages tied at rule score zero",
            "Planned review capacity",
        ],
        "pages": [
            len(lane_slice),
            int(stale_visible.sum()),
            int(fixed_rule_score.eq(0).sum()),
            20,
        ],
    }
)
display(rule_diagnostic)

planned_feature_families = pd.DataFrame(
    {
        "safe pre-decision signal": [
            "impressions_90d",
            "cleaned avg_position",
            "ctr",
            "content_age_days",
            "days_since_last_update",
            "engagement_rate",
        ],
        "reason it may help order ties": [
            "Demand / exposure",
            "Current search visibility",
            "Click capture conditional on visibility",
            "Lifecycle context",
            "Freshness context",
            "Post-click context where measured",
        ],
    }
)
display(planned_feature_families)

forbidden_features = {"trend_direction", "trend_pct", "is_declining_proxy", "content_id", "client_id"}
assert forbidden_features.isdisjoint(planned_feature_families["safe pre-decision signal"])
assert stale_visible.sum() < 20
print("Baseline diagnosis: the rule cannot fill and meaningfully order the 20-page queue by itself.")

,quantity,pages
0,Eligible visible pages,16726
1,Pages flagged by stale-visible rule,17
2,Pages tied at rule score zero,16709
3,Planned review capacity,20


,safe pre-decision signal,reason it may help order ties
0,impressions_90d,Demand / exposure
1,cleaned avg_position,Current search visibility
2,ctr,Click capture conditional on visibility
3,content_age_days,Lifecycle context
4,days_since_last_update,Freshness context
5,engagement_rate,Post-click context where measured


Baseline diagnosis: the rule cannot fill and meaningfully order the 20-page queue by itself.


## 6. Self-check

- [x] The primary task type is named as ranking/scoring.
- [x] The desired future outcome and the starter teaching proxy are separated honestly.
- [x] Precision@20 and a predeclared definition of “good” are stated.
- [x] The output supports a real content strategist and named content actions.
- [x] A real dataframe proves that one row is one unique pseudonymized page.
- [x] The fixed-rule limitation and why ML might help are demonstrated.
- [x] No client names, URLs, private queries, or raw private fields are displayed.
- [x] Claims use careful words: observed, directional, decision-support.

In [6]:
assert starter.shape == (30_000, 45)  # 44 shipped columns + the proxy sketched above
assert lane_slice["content_id"].is_unique
assert len(lane_slice) == 16_726
assert set(starter["is_declining_proxy"].unique()).issubset({0, 1})
assert {"trend_direction", "trend_pct"}.isdisjoint(planned_feature_families["safe pre-decision signal"])
print("SELF-CHECK PASS: task + target/proxy + metric + action + real unit + rule comparison")

SELF-CHECK PASS: task + target/proxy + metric + action + real unit + rule comparison
